# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step example for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We will interact with the Croissant schema using entity `@id` fields to ensure robust referencing and reproducibility.

### Dataset Source
- [Croissant schema (JSON-LD)](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

> **Cite as:** Liu, Y, Duan, X, Yang, S, Zhang, Y, Han, S 2026 Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Frontiers

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. The `@id`s from the Croissant schema are used throughout for referencing entities.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Authors: {getattr(metadata, 'author', None)}")

## 2. Data Overview

Let's review record sets, their fields, and gather their `@id`s for further work.

In [ ]:
# List all record set @id's available in the dataset
record_sets_metadata = dataset.metadata.record_set

if isinstance(record_sets_metadata, list):
    record_sets_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None) for rs in record_sets_metadata]
else:
    record_sets_ids = [record_sets_metadata['@id'] if isinstance(record_sets_metadata, dict) and '@id' in record_sets_metadata else getattr(record_sets_metadata, '@id', None)]
record_sets_ids = [rsid for rsid in record_sets_ids if rsid is not None]

print(f"Record sets in this dataset:")
for i, rsid in enumerate(record_sets_ids):
    print(f"{i+1}. {rsid}")

# For each record set, show its fields by @id
for rsid in record_sets_ids:
    print(f"\nFields for RecordSet '@id': {rsid}")
    # Get RecordSet metadata
    recs = None
    for obj in dataset.metadata.record_set:
        if (isinstance(obj, dict) and obj.get('@id')==rsid) or (hasattr(obj, '@id') and getattr(obj, '@id')==rsid):
            recs = obj
            break
    # List fields by @id
    if recs:
        fields = recs['field'] if isinstance(recs, dict) and 'field' in recs else getattr(recs, 'field', [])
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            fid = field['@id'] if isinstance(field, dict) and '@id' in field else getattr(field, '@id', str(field))
            print(f"  - {fid}")
    else:
        print("  (No field metadata found)")

# Show a small sample record for each record set (referencing by @id)
for rsid in record_sets_ids:
    print(f"\nSample records from record_set {rsid}:")
    try:
        gen = dataset.records(record_set=rsid)
        for i, rec in enumerate(gen):
            if i >= 2:
                break
            print(rec)
    except Exception as e:
        print(f"  - Error reading records: {e}")

## 3. Data Extraction

Load data from available record sets into pandas DataFrames using entity `@id` fields. Use the record set and field `@id`s from above.

In [ ]:
# Collect all dataframes in a dict: {record_set_id: DataFrame}
dataframes = {}
for rsid in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"Loaded DataFrame for record set '@id': {rsid} with {df.shape[0]} records and columns: {df.columns.tolist()}")
        else:
            print(f"No records found in record set '@id': {rsid}")
    except Exception as e:
        print(f"Error loading records for '@id': {rsid}: {e}")

# Choose the main tabular record set (first, or adapt if known)
if len(dataframes) == 0:
    raise ValueError("No dataframes were loaded from the dataset.")

primary_rsid = list(dataframes.keys())[0]
print(f"\nPrimary record set chosen for demonstration: {primary_rsid}")
print("Columns:", dataframes[primary_rsid].columns.tolist())

# Show preview of the main DataFrame
dataframes[primary_rsid].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records and normalizing numeric fields.
The EDA will perform operations referencing each column by its entity `@id` as found in the DataFrame.

> *The actual numeric and groupable field `@id`s may be found in the DataFrame columns above. Update as required for your exploratory tasks!*

In [ ]:
# List columns for user visibility
print("All columns (field @id) in selected record set:")
for col in dataframes[primary_rsid].columns:
    print(f"- {col}")

# Choose a likely numeric field using the column list (edit if needed)
possible_numeric_columns = [col for col in dataframes[primary_rsid].columns if dataframes[primary_rsid][col].dtype in [int, float]]
if not possible_numeric_columns:
    # Try to infer possible numeric fields by casting
    for col in dataframes[primary_rsid].columns:
        try:
            dataframes[primary_rsid][col].astype(float)
            possible_numeric_columns.append(col)
        except Exception:
            continue

if possible_numeric_columns:
    numeric_field_id = possible_numeric_columns[0]
else:
    # Provide an example fallback
    numeric_field_id = dataframes[primary_rsid].columns[0]
    print(f"(No clear numeric column found, using {numeric_field_id} for demonstration.)")
print(f"Numeric field @id selected: {numeric_field_id}")

# Threshold-based filtering
try:
    df_numeric = pd.to_numeric(dataframes[primary_rsid][numeric_field_id], errors='coerce')
except Exception:
    df_numeric = dataframes[primary_rsid][numeric_field_id]
    print("Could not coerce to numeric; using field as-is.")

threshold = 10  # Example threshold
filtered_df = dataframes[primary_rsid][df_numeric > threshold]
print(f"\nFiltered {numeric_field_id} > {threshold}, records: {len(filtered_df)}")
display_cols = [col for col in filtered_df.columns if col==numeric_field_id]
print(filtered_df[display_cols].head())

# Normalize the selected numeric field
if len(filtered_df) > 0:
    filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std(ddof=0)
    print(f"\nTop normalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print('(No records survived filtering; normalization skipped.)')

# Group by a likely categorical field (@id)
possible_categorical = [col for col in dataframes[primary_rsid].columns if col != numeric_field_id and dataframes[primary_rsid][col].dtype == object]
group_field_id = possible_categorical[0] if possible_categorical else None

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable group field for grouping.")

## 5. Visualization

Visualize the filtered and grouped data using matplotlib/seaborn or your preferred tool. All plots will use column `@id` fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (before and after normalization)
if len(filtered_df) > 0:
    plt.figure(figsize=(10,4))
    sns.histplot(pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered > {threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(10,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=10, kde=True, color='orange')
    plt.title(f"Distribution of {numeric_field_id}_normalized")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.ylabel("Frequency")
    plt.show()

    # If grouped, visualize mean per group
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(12,5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, palette='viridis')
        plt.title(f"Mean {numeric_field_id} per {group_field_id}")
        plt.xticks(rotation=45)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization after filtering.")

## 6. Conclusion

In this notebook, we demonstrated loading and exploring the FAIR² clinical colorectal cancer dataset using the `mlcroissant` library and entity referencing via `@id`. We performed:

- Review of the dataset's structure, with all entities referenced by `@id` for reproducibility.
- Data extraction into pandas DataFrames.
- Example exploratory data analysis including filtering, normalization, and grouping, all using `@id` field names.
- Visualization of measurement distributions and group comparisons.

For further analysis, see the detailed field dictionary in the Croissant schema and adapt the EDA to clinical research questions of interest.